In [1]:
import pandas as pd
import mysql.connector

# 1. Notebook Overview

#### Objective

The objective of this notebook is to implement the physical schema of the Data Warehouse in MySQL based on the dimensional model defined during the data preparation stage.

The implementation follows a star-schema approach with conformed dimensions shared across multiple fact tables, allowing efficient analytical queries while maintaining referential integrity and minimizing data redundancy.

Throughout this notebook, the dimensional model is translated into a relational database by defining the database structure, creating dimension and fact tables, specifying primary and foreign key relationships, and validating the resulting schema before the ETL process.

#### Scope

This notebook includes the following tasks:

- Review the final dimensional architecture.
- Define the physical data dictionary for every table.
- Create the MySQL database.
- Implement all dimension tables.
- Implement all fact tables.
- Establish primary and foreign key relationships.
- Validate the final relational schema.

# 2. Prerequisites

#### Objective

Describe the software environment, tools, and technologies required to reproduce the physical implementation of the Data Warehouse presented in this notebook.

#### Software Requirements

The following software was used during the implementation of the relational schema:

|Component|Version|Purpose|
|---------|-------|-------|
|Python|3.x|Data preparation and SQL execution|
|Pandas|2.x|Data manipulation and validation|
|MySQL Server|8.x|Relational Data Warehouse implementation|
|MySQL Workbench|8.x|Database administration and SQL development|
|Jupyter Notebook|Latest|Documentation and implementation|

#### Python Libraries

The following Python libraries are required throughout this notebook:
```python
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
```

#### Database Configuration

The notebook assumes that:

- MySQL Server is installed and running.
- MySQL Workbench is available for database administration.
- The user has sufficient privileges to create databases, tables, and relationships.
- A local MySQL connection is available.

#### Input

The relational schema implemented in this notebook is based on the dimensional model defined in:

- 02_mysql_data_preparation.ipynb

No additional raw datasets are required during this stage since all modeling decisions were completed in the previous notebook.

#### Output

This notebook will generate the complete physical schema of the Oil & Gas Data Warehouse, including:

- Database creation.
- Dimension tables.
- Fact tables.
- Primary keys.
- Foreign keys.
- Referential integrity constraints.

# 3. Data Warehouse Architecture

#### Objective

Present the final dimensional architecture that will be implemented throughout this notebook.

The Data Warehouse follows a fact constellation schema composed of two fact tables sharing a common set of conformed dimensions. This design minimizes data redundancy while supporting analytical queries across production and hydraulic fracturing processes.

#### Implementation

The final schema consists of six shared dimensions and two fact tables.

Dimension Tables

|Dimension|Description|Business Grain|
|---------|-----------|--------------|
|dim_time|Stores the calendar information used by both production and fracturing events.|One calendar date.|
|dim_well|Stores stable physical and operational characteristics of each well.|One physical well.|
|dim_geology|Stores geological attributes associated with each well.|One unique geological profile associated with a well.|
|dim_location|Stores geographical information, including basin, province, and coordinates.|One unique geographical location associated with a well.|
|dim_field_area|Stores concession area and producing field information.|One producing field and concession area combination.|
|dim_company|Stores operating company information.|One operating company.|

Fact Tables

|Fact Table|Granularity|Business Grain|
|----------|-----------|--------------|
|fact_production|One record per well and reporting month.|One monthly production record per well.|
|fact_fracturing|One record per hydraulic fracturing treatment.|One hydraulic fracturing treatment performed on one well.|

#### Star Constellation Diagram

```mermaid
flowchart TB

    DT[dim_time]
    DW[dim_well]
    DG[dim_geology]
    DL[dim_location]
    DFA[dim_field_area]
    DC[dim_company]

    FP[fact_production]
    FF[fact_fracturing]

    DT --> FP
    DT --> FF

    DW --> FP
    DW --> FF

    DG --> FP
    DG --> FF

    DL --> FP
    DL --> FF

    DFA --> FP
    DFA --> FF

    DC --> FP
    DC --> FF
```

#### Validation

The architecture defines a shared dimensional model where all descriptive entities are centralized in conformed dimensions and both business processes reference the same surrogate keys.

This structure guarantees:

- Consistent analytical reporting across production and fracturing data.
- Reduced data redundancy.
- Simplified maintenance of descriptive information.
- Efficient implementation of star-schema queries in MySQL.

# 4. Physical Data Dictionary

#### Objective

Define the physical structure of every table included in the Data Warehouse before implementation in MySQL.

For each dimension and fact table, the data dictionary specifies the column names, data types, keys, nullability, and business descriptions that will guide the physical database creation.

#### Implementation

The following sections describe the complete physical structure of each table included in the dimensional model.

The tables are documented in the same order in which they will later be implemented within the relational schema.

## 4.1 dim_time

#### Objective

Define the physical structure of the shared Time Dimension (dim_time), which will provide a standardized temporal reference for both production and hydraulic fracturing events.

This dimension centralizes all calendar attributes required for analytical reporting while eliminating redundant temporal information stored in the source datasets.

#### Implementation
Business Grain

- One calendar date.

#### Source Tables

- production_sample
- fracture_df

Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|time_key|INT|PK|No|GeneratedSurrogate key uniquely identifying each calendar date.|
|full_date|DATE|-|No|production_sample.fecha_data, fracture_df.fecha_inicio_fractura, fracture_df.fecha_fin_fractura|Calendar date represented by the dimension record.|
|day|TINYINT|-|No|Derived|Day of the month (1–31).|
|day_of_week|TINYINT|-|No|Derived|Numeric representation of the day of the week (1–7), derived from full_date.|
|day_name|VARCHAR(15)|-|No|Derived|Full name of the day of the week (Monday–Sunday), derived from full_date.|
|month|TINYINT|-|No|Derived|Calendar month (1–12).|
|month_name|VARCHAR(15)|-|No|Derived|Full month name (January–December).|
|quarter|TINYINT|-|No|Derived|Calendar quarter (1–4).|
|week_of_year|TINYINT|-|No|Derived|Calendar week number within the year, derived from full_date.|
|year|SMALLINT|-|No|Derived|Calendar year.|


#### Validation

The proposed structure satisfies the temporal requirements identified during the data preparation stage.

The dimension replaces redundant year and month attributes present in the source datasets while providing a single conformed calendar shared by both fact tables.

Business dates from production and fracturing records will reference this dimension through the surrogate key (time_key), ensuring temporal consistency across the entire Data Warehouse.

## 4.2 dim_well

#### Objective

Define the physical structure of the Well Dimension (dim_well), which stores the stable descriptive attributes associated with each well.

This dimension centralizes the permanent characteristics of the wells and avoids redundancy by separating static information from operational attributes that change over time.

#### Implementation

Business Grain
- One well.

Source Tables
- production_sample
- fracture_df

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|well_key|INT|PK|No|Generated|Surrogate key uniquely identifying each well.|
|idpozo|INT|NK|No|production_sample, fracture_df|Natural identifier of the well used in the source systems.|
|sigla|VARCHAR(50)|-|No|production_sample|Official well name or abbreviation used for identification.|
|profundidad|DECIMAL(10,2)|-|Yes|production_sample|Total measured depth of the well.|
|proyecto|VARCHAR(100)|-|Yes|production_sample|Development project to which the well belongs.|
|clasificacion|VARCHAR(50)|-|Yes|production_sample|General classification assigned to the well.|
|subclasificacion|VARCHAR(50)|-|Yes|production_sample|Detailed subclassification associated with the well.|
|longitud_rama_horizontal_m|DECIMAL(10,2)|-|Yes|fracture_df|Horizontal lateral length of the well (meters). Represents a stable physical characteristic of the well and is therefore stored in the Well Dimension rather than in the Fracturing Fact Table.|

#### Validation

The proposed structure contains only stable descriptive attributes that uniquely characterize each well throughout its operational life.

Operational variables such as production type, well status, extraction method, and completion type were intentionally excluded because they may change over time and therefore belong to the corresponding fact tables.

The resulting dimension provides a consistent master reference for both production and hydraulic fracturing analyses while minimizing redundancy across the Data Warehouse.

# 4.3 dim_company

#### Objective

Define the physical structure of the Company Dimension (dim_company), which stores the descriptive information associated with the operating companies responsible for production and hydraulic fracturing activities.

This dimension centralizes company information to ensure consistency and avoid redundancy across the Data Warehouse.

#### Implementation

Business Grain

- One company.

Source Tables

- production_sample
- fracturing_df

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|company_key|INT|PK|No|Generated|Surrogate key uniquely identifying each company.|
|idempresa|VARCHAR(20)|NK|No|production_sample|Natural identifier of the operating company in the source system.|
|empresa|VARCHAR(150)|-|No|production_sample, fracturing_df|Official name of the operating company.|

#### Validation

The proposed structure contains the stable descriptive attributes that uniquely identify each operating company.

The surrogate key (company_key) will be used as the foreign key in the fact tables, while idempresa is preserved as the natural key from the source system for traceability and data integration purposes.

The resulting dimension provides a standardized company reference for analytical reporting while eliminating duplicated company information across the Data Warehouse.

# 4.4 dim_geology

#### Objective

Define the physical structure of the Geology Dimension (dim_geology), which stores the stable geological characteristics associated with each well.

This dimension centralizes geological information used for reservoir characterization and analytical segmentation while eliminating redundancy across the production and hydraulic fracturing datasets.

#### Implementation

Business Grain

- One unique geological characterization of a well.

Source Tables

- production_sample
- fracture_df

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|geology_key|INT|PK|No|Generated|Surrogate key uniquely identifying each geological profile.|
|formacion|VARCHAR(100)|-|Yes|production_sample|Geological formation associated with the well.|
|subtipo_recurso|VARCHAR(50)|-|Yes|production_sample.subtipo_recurso|Subtype of hydrocarbon resource (e.g., SHALE, TIGHT).|
|tipo_reservorio|VARCHAR(50)|-|Yes|fracture_df|General reservoir classification associated with the well.|
|subtipo_reservorio|VARCHAR(50)|-|Yes|fracture_df|Detailed reservoir subtype associated with the well.|
|formacion_productiva|VARCHAR(100)|-|Yes|fracture_df|Productive geological formation targeted during hydraulic fracturing.|

#### Validation

The dimension integrates stable geological attributes obtained from both the production and hydraulic fracturing datasets. Although the attributes originate from different operational sources, they describe the same geological characteristics of each well and remain stable over time.

Redundant attributes such as formprod were excluded because they duplicate information already represented by formacion, while constant attributes such as tipo_de_recurso were intentionally removed because they provide no analytical differentiation.

The resulting dimension provides a standardized geological reference that supports production, reservoir, and hydraulic fracturing analyses without introducing redundancy into the dimensional model.

# 4.5 dim_field_area

#### Objective

Define the physical structure of the Field Area Dimension (dim_field_area), which stores the stable administrative information related to concession areas and producing fields associated with each well.

This dimension centralizes field-level information to support geographical and operational analyses while eliminating redundancy across the Data Warehouse.

#### Implementation

Business Grain

- One unique concession area and producing field combination.

Source Tables

- production_sample
- fracture_df

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|field_area_key|INT|PK|No|Generated|Surrogate key uniquely identifying each concession area and producing field combination.|
|idareapermisoconcesion|VARCHAR(20)|NK|No|production_sample|Natural identifier of the concession area.|
|areapermisoconcesion|VARCHAR(150)|-|No|production_sample|Name of the concession area associated with the well.|
|idareayacimiento|VARCHAR(20)|NK|No|production_sample|Natural identifier of the producing field area.|
|areayacimiento|VARCHAR(150)|-|No|production_sample|Name of the producing field area.|
|yacimiento|VARCHAR(150)|-|Yes|fracture_df|Producing field associated with the well, used for analytical grouping and field-level reporting.|

#### Validation

The proposed structure contains only stable administrative attributes describing the concession area and producing field associated with each well.

These attributes remain constant over time and therefore belong in a dimension rather than a fact table.

The resulting dimension provides a standardized reference for spatial and operational analyses while avoiding repeated field information across the production and hydraulic fracturing fact tables.

# 4.6 dim_location

#### Objective

Define the physical structure of the Location Dimension (dim_location), which stores the stable geographical attributes associated with each well.

This dimension centralizes spatial information used for regional analysis and geographical segmentation while eliminating redundancy across the Data Warehouse.

#### Implementation

Business Grain

- One unique geographical location of a well.

Source Tables

- production_sample

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|location_key|INT|PK|No|Generated|Surrogate key uniquely identifying each geographical location.|
|cuenca|VARCHAR(100)|-|No|production_sample|Sedimentary basin where the well is located.|
|provincia|VARCHAR(100)|-|No|production_sample|Argentine province where the well is located.|
|coordenadax|DECIMAL(12,6)|-|Yes|production_sample|East-West geographic coordinate (X) of the well location.|
|coordenaday|DECIMAL(12,6)|-|Yes|production_sample|North-South geographic coordinate (Y) of the well location.|

#### Validation

The proposed structure contains only stable geographical attributes describing the physical location of each well.

The coordinates provide precise spatial positioning, while cuenca and provincia support higher-level geographical aggregation for regional analyses.

The resulting dimension establishes a standardized geographical reference that enables spatial reporting, regional comparisons, and basin-level production analyses without duplicating location information across the fact tables.

# 4.7 fact_production

#### Objective

Define the physical structure of the Production Fact Table (fact_production), which stores the monthly production measurements recorded for each well.

This fact table centralizes the quantitative production metrics together with the operational attributes that may vary over time, allowing analytical reporting at the monthly well level.

#### Implementation

Business Grain

- One production record for one well during one reporting month.

Source Tables

- production_sample

#### Foreign Keys

|Foreign Key|References|
|-----------|----------|
|time_key|dim_time|
|well_key|dim_well|
|company_key|dim_company|
|geology_key|dim_geology|
|location_key|dim_location|
|field_area_key|dim_field_area|

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|production_key|INT|PK|No|Generated|Surrogate key uniquely identifying each production record.|
|time_key|INT|FK|No|dim_time|References the reporting month of the production record.|
|well_key|INT|FK|No|dim_well|References the producing well.|
|company_key|INT|FK|No|dim_company|References the operating company.|
|geology_key|INT|FK|No|dim_geology|References the geological characteristics of the well.|
|location_key|INT|FK|No|dim_location|References the geographical location of the well.|
|field_area_key|INT|FK|No|dim_field_area|References the concession area and producing field associated with the well.|
|prod_pet|DECIMAL(12,2)|-|No|production_sample|Monthly oil production.|
|prod_gas|DECIMAL(12,2)|-|No|production_sample|Monthly natural gas production.|
|prod_agua|DECIMAL(12,2)|-|No|production_sample|Monthly water production.|
|iny_agua|DECIMAL(12,2)|-|No|production_sample|Monthly injected water volume.|
|tef|DECIMAL(8,2)|-|No|production_sample|Effective operating time during the reporting period.|
|tipoextraccion|VARCHAR(50)|-|Yes|production_sample|Extraction method used during the reporting period.|
|tipoestado|VARCHAR(50)|-|Yes|production_sample|Operational status of the well during the reporting period.|
|tipopozo|VARCHAR(50)|-|Yes|production_sample|Well type recorded during the reporting period.|

#### Validation

The proposed structure follows a star schema in which every production record is linked to the corresponding dimensions through surrogate foreign keys.

Quantitative production measures and time-dependent operational attributes are stored in the fact table, while all stable descriptive information is maintained within the dimension tables.

This design minimizes redundancy, simplifies analytical queries, and provides a scalable structure for future extensions of the Data Warehouse.

# 4.8 fact_fracturing

#### Objective

Define the physical structure of the Hydraulic Fracturing Fact Table (fact_fracturing), which stores the operational measurements recorded during each hydraulic fracturing treatment.

This fact table centralizes treatment metrics together with the corresponding dimensional references, enabling detailed analyses of hydraulic fracturing operations across wells, geological formations, locations, and time.

#### Implementation

Business Grain

- One hydraulic fracturing treatment performed on one well.

Source Tables

- fracture_df

#### Foreign Keys

|Foreign Key|References|
|-----------|----------|
|time_key|dim_time|
|well_key|dim_well|
|company_key|dim_company|
|geology_key|dim_geology|
|location_key|dim_location|
|field_area_key|dim_field_area|

#### Physical Data Dictionary

|Column|Data Type|Key|Nullable|Source|Description|
|------|---------|---|--------|------|-----------|
|fracturing_key|INT|PK|No|Generated|Surrogate key uniquely identifying each hydraulic fracturing treatment.|
|time_key|INT|FK|No|dim_time|References the hydraulic fracturing date.|
|well_key|INT|FK|No|dim_well|References the fractured well.|
|company_key|INT|FK|No|dim_company|References the operating company.|
|geology_key|INT|FK|No|dim_geology|References the geological characteristics of the well.|
|location_key|INT|FK|No|dim_location|References the geographical location of the well.|
|field_area_key|INT|FK|No|dim_field_area|References the concession area and producing field associated with the well.|
|arena_bombeada_nacional_tn|DECIMAL(12,2)|-|No|fracture_df|National proppant pumped during the treatment (tons).|
|arena_bombeada_importada_tn|DECIMAL(12,2)|-|No|fracture_df|Imported proppant pumped during the treatment (tons).|
|agua_inyectada_m3|DECIMAL(12,2)|-|No|fracture_df|Water injected during the hydraulic fracturing treatment (m³).|
|co2_inyectado_m3|DECIMAL(12,2)|-|No|fracture_df|CO₂ injected during the hydraulic fracturing treatment (m³).|
|presion_maxima_psi|DECIMAL(12,2)|-|No|fracture_df|Maximum pumping pressure reached during the treatment (psi).|
|potencia_equipos_fractura_hp|DECIMAL(12,2)|-|Yes|fracture_df|Total pumping horsepower available during the treatment.|
|cantidad_fracturas|INT|-|No|fracture_df|Number of fracturing stages performed during the treatment.|

#### Validation

The proposed structure follows the same star schema adopted for fact_production, linking every hydraulic fracturing treatment to the corresponding dimensions through surrogate foreign keys.

Only quantitative operational measurements are stored in the fact table, while stable descriptive information remains in the dimension tables. This design minimizes redundancy, simplifies analytical queries, and provides a scalable structure for evaluating hydraulic fracturing treatments across different wells, geological settings, geographical locations, and time periods.

# 5. Database Creation

#### Objective

Create the MySQL database that will host the Oil & Gas Data Warehouse.

The database provides the physical environment where the dimensional model designed in the previous sections will be implemented, including all dimension tables, fact tables, primary keys, foreign keys, and supporting indexes.

#### Implementation

Database Definition

The Data Warehouse will be implemented as a dedicated MySQL database following the star schema designed throughout this notebook.

The database will contain:

- Six dimension tables (dim_time, dim_well, dim_company, dim_geology, dim_location, and dim_field_area).
- Two fact tables (fact_production and fact_fracturing).
- Surrogate primary keys for every dimension and fact table.
- Foreign key relationships enforcing referential integrity across the star schema.

### Database Creation

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)

In [3]:
def execute_sql(sql, message = None):
    try:
        cursor.execute(sql)
        conn.commit()
        if message:
            print(message)
    except mysql.connector.Error as err:
        print(f"Error: {err}")

#### Script

```sql
CREATE DATABASE IF NOT EXISTS oil_gas_dw
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci;

USE oil_gas_dw;

```

#### Execute SQL

In [5]:
sql = """
CREATE DATABASE IF NOT EXISTS oil_gas_dw
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci;
"""

execute_sql(sql, "Database created successfully")

Database created successfully


In [4]:
cursor.execute("USE oil_gas_dw")
print("Database selected")

Database selected


### Validation

The successful execution of the script creates an empty database that will serve as the physical container for the dimensional model.

At this stage, no tables are created yet. The database only establishes the environment where the dimension tables, fact tables, constraints, and indexes will be implemented in the following sections.

# 6. Dimension Tables Creation

#### Objective

Implement the physical dimension tables of the Oil & Gas Data Warehouse in MySQL.

Each dimension table is created according to the dimensional model defined in the previous sections, using surrogate primary keys and appropriate SQL data types to support analytical queries while maintaining data integrity and scalability.

#### Implementation

The dimension tables are created following the star schema architecture previously designed.

Each table includes:

- A surrogate primary key.
- Business attributes describing the corresponding entity.
- Appropriate MySQL data types.
- Primary key constraints.
- Natural keys when applicable.

The implementation follows the same order used during the dimensional modeling process.

#### Validation

After executing the SQL scripts, all dimension tables should exist within the oil_gas_dw database.

Each table must contain the expected columns, primary key definitions, and data types established in the Physical Data Dictionary.

Successful creation of these dimensions provides the foundation required for implementing the fact tables and enforcing referential integrity in subsequent steps.

## 6.1 dim_time

### Objective

Implement the Time Dimension (dim_time) in MySQL.

This dimension provides a standardized calendar structure shared by all fact tables, allowing consistent temporal analysis across production and hydraulic fracturing records.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_time(
    time_key INT AUTO_INCREMENT PRIMARY KEY,
    full_date DATE NOT NULL UNIQUE,
    year SMALLINT NOT NULL,
    quarter TINYINT NOT NULL,
    month TINYINT NOT NULL,
    month_name VARCHAR(15) NOT NULL,
    day TINYINT NOT NULL,
    day_of_week TINYINT NOT NULL,
    day_name VARCHAR(15) NOT NULL,
    week_of_year TINYINT NOT NULL
);
```

#### Execute SQL

In [7]:
dim_time = """
CREATE TABLE dim_time(
    time_key INT AUTO_INCREMENT PRIMARY KEY,
    full_date DATE NOT NULL UNIQUE,
    year SMALLINT NOT NULL,
    quarter TINYINT NOT NULL,
    month TINYINT NOT NULL,
    month_name VARCHAR(15) NOT NULL,
    day TINYINT NOT NULL,
    day_of_week TINYINT NOT NULL,
    day_name VARCHAR(15) NOT NULL,
    week_of_year TINYINT NOT NULL
);
"""

execute_sql(dim_time, "dim_table created successfully")

dim_table created successfully


### Validation

After executing the script, verify that the table has been created successfully and that all columns match the expected structure.

In [5]:
def execute_query(sql):
    cursor.execute(sql)
    return cursor.fetchall()

In [6]:
def describe_table(table_name):
    return pd.DataFrame(
        execute_query(f"DESCRIBE {table_name}"),
        columns=[
            "Field",
            "Type",
            "Null",
            "Key",
            "Default",
            "Extra"
        ]
    )

In [7]:
def show_create_table(table_name):
    return execute_query(f"SHOW CREATE TABLE {table_name}")[0][1]

```sql
DESCRIBE dim_time
```

```sql
SHOW CREATE TABLE dim_time;
```

#### Table Structure

In [11]:
describe_table("dim_time")

,Field,Type,Null,Key,Default,Extra
0,time_key,int,NO,PRI,None,auto_increment
1,full_date,date,NO,UNI,None,
2,year,smallint,NO,,None,
3,quarter,tinyint,NO,,None,
4,month,tinyint,NO,,None,
5,month_name,varchar(15),NO,,None,
6,day,tinyint,NO,,None,
7,day_of_week,tinyint,NO,,None,
8,day_name,varchar(15),NO,,None,
9,week_of_year,tinyint,NO,,None,


#### SQL Definition

In [12]:
show_create_table("dim_time")

'CREATE TABLE `dim_time` (\n  `time_key` int NOT NULL AUTO_INCREMENT,\n  `full_date` date NOT NULL,\n  `year` smallint NOT NULL,\n  `quarter` tinyint NOT NULL,\n  `month` tinyint NOT NULL,\n  `month_name` varchar(15) COLLATE utf8mb4_unicode_ci NOT NULL,\n  `day` tinyint NOT NULL,\n  `day_of_week` tinyint NOT NULL,\n  `day_name` varchar(15) COLLATE utf8mb4_unicode_ci NOT NULL,\n  `week_of_year` tinyint NOT NULL,\n  PRIMARY KEY (`time_key`),\n  UNIQUE KEY `full_date` (`full_date`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## 6.2 dim_well

### Objective

Implement the Well Dimension (dim_well) in MySQL.

This dimension stores the stable physical and descriptive characteristics of each well. It centralizes well-level information that remains constant over time and serves as a shared reference for both the Production and Hydraulic Fracturing fact tables.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_well (
    well_key INT AUTO_INCREMENT PRIMARY KEY,
    idpozo INT NOT NULL UNIQUE,
    sigla VARCHAR(50),
    profundidad DECIMAL(10,2)
    longitd_rama_horizontal DECIMAL(10,2),
    proyecto VARCHAR(100),
    clasificacion VARCHAR(100),
    subclasificacion VARCHAR(100)
)
```

#### Execute SQL

In [ ]:
dim_well = """
CREATE TABLE dim_well(
    well_key INT AUTO_INCREMENT PRIMARY KEY,
    idpozo INT NOT NULL UNIQUE,
    sigla VARCHAR(50),
    profundidad DECIMAL(10,2),
    longitud_rama_horizontal_m DECIMAL(10,2),
    proyecto VARCHAR(100),
    clasificacion VARCHAR(100),
    subclasificacion VARCHAR(100)
)
"""

execute_sql(dim_well, "dim_well created successfully")

dim_well created successfully


### Validation

#### Table Structure

In [14]:
describe_table("dim_well")

,Field,Type,Null,Key,Default,Extra
0,well_key,int,NO,PRI,None,auto_increment
1,idpozo,int,NO,UNI,None,
2,sigla,varchar(50),NO,,None,
3,profundidad,"decimal(10,2)",YES,,None,
4,longitud_rama_horizontal_m,"decimal(10,2)",YES,,None,
5,proyecto,varchar(100),YES,,None,
6,clasificacion,varchar(100),YES,,None,
7,subclasificacion,varchar(100),YES,,None,


#### SQL Definition

In [15]:
show_create_table("dim_well")

'CREATE TABLE `dim_well` (\n  `well_key` int NOT NULL AUTO_INCREMENT,\n  `idpozo` int NOT NULL,\n  `sigla` varchar(50) COLLATE utf8mb4_unicode_ci NOT NULL,\n  `profundidad` decimal(10,2) DEFAULT NULL,\n  `longitud_rama_horizontal_m` decimal(10,2) DEFAULT NULL,\n  `proyecto` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `clasificacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `subclasificacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  PRIMARY KEY (`well_key`),\n  UNIQUE KEY `idpozo` (`idpozo`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## 6.3 dim_geology

### Objective

Implement the Geological Dimension (dim_geology) in MySQL.

This dimension stores the stable geological characteristics associated with each well. It centralizes formation and reservoir information that supports geological segmentation and enables analytical comparisons across production and hydraulic fracturing activities.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_geology(
    geology_key INT AUTO_INCREMENT PRIMARY KEY,
    formacion VARCHAR(100),
    subtipo_recurso VARCHAR(50),
    tipo_reservorio VARCHAR(50)
    subtipo_reservorio VARCHAR(50),
    formacion_productiva VARCHAR(100)
)
```

#### Execute SQL

In [9]:
dim_geology = """
CREATE TABLE dim_geology(
    geology_key INT AUTO_INCREMENT PRIMARY KEY,
    formacion VARCHAR(100),
    subtipo_recurso VARCHAR(50),
    tipo_reservorio VARCHAR(50),
    subtipo_reservorio VARCHAR(50),
    formacion_productiva VARCHAR(100)
)
"""

execute_sql(dim_geology, "dim_geology created successfully")

dim_geology created successfully


### Validation

#### Table Structure

In [17]:
describe_table("dim_geology")

,Field,Type,Null,Key,Default,Extra
0,geology_key,int,NO,PRI,None,auto_increment
1,formacion,varchar(100),YES,,None,
2,subtipo_recurso,varchar(50),YES,,None,
3,tipo_reservorio,varchar(50),YES,,None,
4,subtipo_reservorio,varchar(50),YES,,None,
5,formacion_productiva,varchar(100),YES,,None,


#### SQL Definition

In [18]:
show_create_table("dim_geology")

'CREATE TABLE `dim_geology` (\n  `geology_key` int NOT NULL AUTO_INCREMENT,\n  `formacion` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `subtipo_recurso` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `tipo_reservorio` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `subtipo_reservorio` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `formacion_productiva` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  PRIMARY KEY (`geology_key`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## 6.4 dim_field_area

### Objective

Implement the Field Area Dimension (dim_field_area) in MySQL.

This dimension stores the stable administrative and operational area information associated with each well. It centralizes concession areas and producing fields, enabling analyses at different geographic and operational aggregation levels.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_field_area(
    field_area_key INT AUTO_INCREMENT PRIMARY KEY,
    idareapermisoconcesion VARCHAR(20),
    areapermisoconcesion VARCHAR(150),
    idareayacimiento VARCHAR(20),
    areayacimiento VARCHAR(150),
    yacimiento VARCHAR(150)
);
```

#### Execute SQL

In [12]:
dim_field_area = """
CREATE TABLE dim_field_area(
    field_area_key INT AUTO_INCREMENT PRIMARY KEY,
    idareapermisoconcesion VARCHAR(20),
    areapermisoconcesion VARCHAR(150),
    idareayacimiento VARCHAR(20),
    areayacimiento VARCHAR(150),
    yacimiento VARCHAR(150)
)
"""

execute_sql(dim_field_area, "dim_field_area created successfully")

dim_field_area created successfully


### Validation

#### Table Structure

In [20]:
describe_table("dim_field_area")

,Field,Type,Null,Key,Default,Extra
0,field_area_key,int,NO,PRI,None,auto_increment
1,idareapermisoconcesion,int,YES,,None,
2,areapermisoconcesion,varchar(150),YES,,None,
3,idareayacimiento,int,YES,,None,
4,areayacimiento,varchar(150),YES,,None,
5,yacimiento,varchar(150),YES,,None,


#### SQL Definition

In [21]:
show_create_table("dim_field_area")

'CREATE TABLE `dim_field_area` (\n  `field_area_key` int NOT NULL AUTO_INCREMENT,\n  `idareapermisoconcesion` int DEFAULT NULL,\n  `areapermisoconcesion` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `idareayacimiento` int DEFAULT NULL,\n  `areayacimiento` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `yacimiento` varchar(150) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  PRIMARY KEY (`field_area_key`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## 6.5 dim_location

### Objective

Implement the Location Dimension (dim_location) in MySQL.

This dimension stores the stable geographic attributes associated with each well. It centralizes regional and spatial information, enabling geographic analyses and supporting location-based aggregation across production and hydraulic fracturing activities.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_location(
    location_key INT AUTO_INCREMENT PRIMARY KEY,
    cuenca VARCHAR(100),
    provincia VARCHAR(100),
    coordenadax DECIMAL(12,8)
    coordenaday DECIMAL(12,8)
);
```

#### Execute SQL

In [8]:
dim_location = """
CREATE TABLE dim_location(
    location_key INT AUTO_INCREMENT PRIMARY KEY,
    cuenca VARCHAR(100),
    provincia VARCHAR(100),
    coordenadax DECIMAL(12,8),
    coordenaday DECIMAL(12,8)
)
"""

execute_sql(dim_location, "dim_location created successfully")

dim_location created successfully


### Validation

#### Table Structure

In [23]:
describe_table("dim_location")

,Field,Type,Null,Key,Default,Extra
0,location_key,int,NO,PRI,None,auto_increment
1,cuenca,varchar(100),YES,,None,
2,provincia,varchar(100),YES,,None,
3,coordenadax,"decimal(12,8)",YES,,None,
4,coordenaday,"decimal(12,8)",YES,,None,


#### SQL Definition

In [24]:
show_create_table("dim_location")

'CREATE TABLE `dim_location` (\n  `location_key` int NOT NULL AUTO_INCREMENT,\n  `cuenca` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `provincia` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `coordenadax` decimal(12,8) DEFAULT NULL,\n  `coordenaday` decimal(12,8) DEFAULT NULL,\n  PRIMARY KEY (`location_key`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## 6.6 dim_company

### Objective

Implement the Company Dimension (dim_company) in MySQL.

This dimension stores the operating companies associated with the wells. It centralizes company information, allowing production and hydraulic fracturing activities to be analyzed by operator while avoiding redundancy across the fact tables.

### Implementation

#### SQL Script

```sql
CREATE TABLE dim_company(
    company_key INT AUTO_INCREMENT PRIMARY KEY,
    idempresa VARCHAR(20) NOT NULL UNIQUE,
    empresa VARCHAR(150) NOT NULL
);
```

#### Execute SQL

In [6]:
dim_company = """
CREATE TABLE dim_company(
    company_key INT AUTO_INCREMENT PRIMARY KEY,
    idempresa VARCHAR(20) NOT NULL UNIQUE,
    empresa VARCHAR(150) NOT NULL
);
"""

execute_sql(dim_company, "dim_company created successfully")

dim_company created successfully


### Validation

#### Table Structure

In [26]:
describe_table("dim_company")

,Field,Type,Null,Key,Default,Extra
0,company_key,int,NO,PRI,None,auto_increment
1,idempresa,int,NO,UNI,None,
2,empresa,varchar(150),NO,,None,


#### SQL Definition

In [27]:
show_create_table("dim_company")

'CREATE TABLE `dim_company` (\n  `company_key` int NOT NULL AUTO_INCREMENT,\n  `idempresa` int NOT NULL,\n  `empresa` varchar(150) COLLATE utf8mb4_unicode_ci NOT NULL,\n  PRIMARY KEY (`company_key`),\n  UNIQUE KEY `idempresa` (`idempresa`)\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci'

## Conclusion

All dimension tables have been successfully created following the dimensional model defined during the data preparation phase. Each dimension uses a surrogate primary key to ensure consistency and independence from the source systems. With the shared dimensions in place, the data warehouse is now ready for the implementation of the Production and Hydraulic Fracturing fact tables.

# 7 Fact Tables Creation

## Notebook Overview

This section implements the fact tables of the dimensional model.

Each fact table stores the quantitative business measures together with the foreign keys that establish relationships with the corresponding dimensions. The implementation follows the star schema designed during the data preparation phase, ensuring referential integrity, analytical flexibility, and consistency across the data warehouse.

## Objective

Create the two central fact tables of the data warehouse:

- fact_production, containing monthly production measurements for each well.
- fact_fracturing, containing operational measurements associated with each hydraulic fracturing treatment.

Both tables will reference the previously created dimensions through surrogate keys

## Implementation Strategy

The fact tables will be implemented following these principles:

- Every fact table will contain only foreign keys referencing the dimension tables.
- Business measures will remain in the fact tables.
- Stable descriptive attributes have already been moved to their corresponding dimensions.
- Referential integrity will be enforced through foreign key constraints.
- Surrogate keys will be used for every dimensional relationship.

## 7.1 fact_production

### Objective

Implement the Production Fact Table (fact_production) in MySQL.

This table stores the monthly production measurements for each well together with the foreign keys that reference the corresponding dimensions. It represents the central analytical table for production reporting and supports multidimensional analysis across time, wells, geology, location, field areas, and operating companies.

### Implementation

#### SQL Script

```sql
CREATE TABLE fact_production (
    production_key INT AUTO_INCREMENT PRIMARY KEY,
    time_key INT NOT NULL,
    well_key INT NOT NULL,
    geology_key INT NOT NULL,
    field_area_key INT NOT NULL,
    location_key INT NOT NULL,
    company_key INT NOT NULL,
    prod_pet DECIMAL(15,2),
    prod_gas DECIMAL(15,2),
    prod_agua DECIMAL(15,2),
    iny_agua DECIMAL(15,2),
    tef DECIMAL(10,2),
    tipoextraccion VARCHAR(50),
    tipoestado VARCHAR(50),
    tipopozo VARCHAR(50),

    CONSTRAINT uq_fact_production
        UNIQUE (time_key, well_key),

    CONSTRAINT fk_fact_production_time
        FOREIGN KEY (time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_production_well
        FOREIGN KEY (well_key)
        REFERENCES dim_well(well_key),

    CONSTRAINT fk_fact_production_geology
        FOREIGN KEY (geology_key)
        REFERENCES dim_geology(geology_key),

    CONSTRAINT fk_fact_production_field_area
        FOREIGN KEY (field_area_key)
        REFERENCES dim_field_area(field_area_key),

    CONSTRAINT fk_fact_production_location
        FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    CONSTRAINT fk_fact_production_company
        FOREIGN KEY (company_key)
        REFERENCES dim_company(company_key)
);

```

#### Execute SQL

In [10]:
fact_production = """
CREATE TABLE fact_production (
    production_key INT AUTO_INCREMENT PRIMARY KEY,
    time_key INT NOT NULL,
    well_key INT NOT NULL,
    geology_key INT NOT NULL,
    field_area_key INT NOT NULL,
    location_key INT NOT NULL,
    company_key INT NOT NULL,
    prod_pet DECIMAL(15,2),
    prod_gas DECIMAL(15,2),
    prod_agua DECIMAL(15,2),
    iny_agua DECIMAL(15,2),
    tef DECIMAL(10,2),
    tipoextraccion VARCHAR(50),
    tipoestado VARCHAR(50),
    tipopozo VARCHAR(50),

    CONSTRAINT uq_fact_production
        UNIQUE (time_key, well_key),

    CONSTRAINT fk_fact_production_time
        FOREIGN KEY (time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_production_well
        FOREIGN KEY (well_key)
        REFERENCES dim_well(well_key),

    CONSTRAINT fk_fact_production_geology
        FOREIGN KEY (geology_key)
        REFERENCES dim_geology(geology_key),

    CONSTRAINT fk_fact_production_field_area
        FOREIGN KEY (field_area_key)
        REFERENCES dim_field_area(field_area_key),

    CONSTRAINT fk_fact_production_location
        FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    CONSTRAINT fk_fact_production_company
        FOREIGN KEY (company_key)
        REFERENCES dim_company(company_key)

);
"""

execute_sql(fact_production, "fact_production created successfully.")

fact_production created successfully.


### Validation

#### Table Structure

In [ ]:
describe_table("fact_production")

,Field,Type,Null,Key,Default,Extra
0,production_key,int,NO,PRI,None,auto_increment
1,time_key,int,NO,MUL,None,
2,well_key,int,NO,MUL,None,
3,geology_key,int,NO,MUL,None,
4,field_area_key,int,NO,MUL,None,
5,location_key,int,NO,MUL,None,
6,company_key,int,NO,MUL,None,
7,prod_pet,"decimal(15,2)",YES,,None,
8,prod_gas,"decimal(15,2)",YES,,None,
9,prod_agua,"decimal(15,2)",YES,,None,


#### SQL Definition

In [30]:
show_create_table("fact_production")

'CREATE TABLE `fact_production` (\n  `production_key` int NOT NULL AUTO_INCREMENT,\n  `time_key` int NOT NULL,\n  `well_key` int NOT NULL,\n  `geology_key` int NOT NULL,\n  `field_area_key` int NOT NULL,\n  `location_key` int NOT NULL,\n  `company_key` int NOT NULL,\n  `prod_pet` decimal(15,2) DEFAULT NULL,\n  `prod_gas` decimal(15,2) DEFAULT NULL,\n  `prod_agua` decimal(15,2) DEFAULT NULL,\n  `iny_agua` decimal(15,2) DEFAULT NULL,\n  `tef` decimal(10,2) DEFAULT NULL,\n  `tipoextraccion` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `tipoestado` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  `tipopozo` varchar(50) COLLATE utf8mb4_unicode_ci DEFAULT NULL,\n  PRIMARY KEY (`production_key`),\n  UNIQUE KEY `uq_fact_production` (`time_key`,`well_key`),\n  KEY `fk_fact_production_well` (`well_key`),\n  KEY `fk_fact_production_geology` (`geology_key`),\n  KEY `fk_fact_production_field_area` (`field_area_key`),\n  KEY `fk_fact_production_location` (`location_key`),\n  KEY `f

## 7.2 fact_fracturing

### Objective

Implement the Hydraulic Fracturing Fact Table (fact_fracturing) in MySQL.

This table stores the operational measurements associated with each hydraulic fracturing treatment together with the foreign keys that reference the corresponding dimensions. Unlike monthly production records, each fracturing event is represented by its own operational dates and treatment characteristics.

### Implementation

#### Temporal Relationships

Unlike fact_production, each hydraulic fracturing treatment contains two business dates:

- the start date of the treatment;
- the end date of the treatment.

Both dates are analytically relevant and therefore reference the shared dim_time table.

start_time_key  → dim_time

end_time_key    → dim_time

#### SQL Script

```sql
CREATE TABLE fact_fracturing (

    fracturing_key INT AUTO_INCREMENT PRIMARY KEY,
    id_base_fractura_adjiv INT NOT NULL UNIQUE,
    start_time_key INT NOT NULL,
    end_time_key INT NOT NULL,
    well_key INT NOT NULL,
    geology_key INT NOT NULL,
    field_area_key INT NOT NULL,
    location_key INT NOT NULL,
    company_key INT NOT NULL,
    arena_bombeada_nacional_tn DECIMAL(15,2),
    arena_bombeada_importada_tn DECIMAL(15,2),
    agua_inyectada_m3 DECIMAL(15,2),
    co2_inyectado_m3 DECIMAL(15,2),
    presion_maxima_psi DECIMAL(10,2),
    potencia_equipos_fractura_hp DECIMAL(10,2),
    cantidad_fracturas SMALLINT,

    CONSTRAINT fk_fact_fracturing_start_time
        FOREIGN KEY (start_time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_fracturing_end_time
        FOREIGN KEY (end_time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_fracturing_well
        FOREIGN KEY (well_key)
        REFERENCES dim_well(well_key),

    CONSTRAINT fk_fact_fracturing_geology
        FOREIGN KEY (geology_key)
        REFERENCES dim_geology(geology_key),

    CONSTRAINT fk_fact_fracturing_field_area
        FOREIGN KEY (field_area_key)
        REFERENCES dim_field_area(field_area_key),

    CONSTRAINT fk_fact_fracturing_location
        FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    CONSTRAINT fk_fact_fracturing_company
        FOREIGN KEY (company_key)
        REFERENCES dim_company(company_key)

);

```

#### Execute SQL

In [11]:
fact_fracturing = """
CREATE TABLE fact_fracturing (
    fracturing_key INT AUTO_INCREMENT PRIMARY KEY,
    id_base_fractura_adjiv INT NOT NULL UNIQUE,
    start_time_key INT NOT NULL,
    end_time_key INT NOT NULL,
    well_key INT NOT NULL,
    geology_key INT NOT NULL,
    field_area_key INT NOT NULL,
    location_key INT NOT NULL,
    company_key INT NOT NULL,
    arena_bombeada_nacional_tn DECIMAL(15,2),
    arena_bombeada_importada_tn DECIMAL(15,2),
    agua_inyectada_m3 DECIMAL(15,2),
    co2_inyectado_m3 DECIMAL(15,2),
    presion_maxima_psi DECIMAL(10,2),
    potencia_equipos_fractura_hp DECIMAL(10,2),
    cantidad_fracturas SMALLINT,

    CONSTRAINT fk_fact_fracturing_start_time
        FOREIGN KEY (start_time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_fracturing_end_time
        FOREIGN KEY (end_time_key)
        REFERENCES dim_time(time_key),

    CONSTRAINT fk_fact_fracturing_well
        FOREIGN KEY (well_key)
        REFERENCES dim_well(well_key),

    CONSTRAINT fk_fact_fracturing_geology
        FOREIGN KEY (geology_key)
        REFERENCES dim_geology(geology_key),

    CONSTRAINT fk_fact_fracturing_field_area
        FOREIGN KEY (field_area_key)
        REFERENCES dim_field_area(field_area_key),

    CONSTRAINT fk_fact_fracturing_location
        FOREIGN KEY (location_key)
        REFERENCES dim_location(location_key),

    CONSTRAINT fk_fact_fracturing_company
        FOREIGN KEY (company_key)
        REFERENCES dim_company(company_key)

);
"""

execute_sql(fact_fracturing, "fact_fracturing created successfully.")

fact_fracturing created successfully.


### Validation

#### Table Structure

In [32]:
describe_table("fact_fracturing")

,Field,Type,Null,Key,Default,Extra
0,fracturing_key,int,NO,PRI,None,auto_increment
1,id_base_fractura_adjiv,int,NO,UNI,None,
2,start_time_key,int,NO,MUL,None,
3,end_time_key,int,NO,MUL,None,
4,well_key,int,NO,MUL,None,
5,geology_key,int,NO,MUL,None,
6,field_area_key,int,NO,MUL,None,
7,location_key,int,NO,MUL,None,
8,company_key,int,NO,MUL,None,
9,arena_bombeada_nacional_tn,"decimal(15,2)",YES,,None,


#### SQL Definition

In [33]:
show_create_table("fact_fracturing")

'CREATE TABLE `fact_fracturing` (\n  `fracturing_key` int NOT NULL AUTO_INCREMENT,\n  `id_base_fractura_adjiv` int NOT NULL,\n  `start_time_key` int NOT NULL,\n  `end_time_key` int NOT NULL,\n  `well_key` int NOT NULL,\n  `geology_key` int NOT NULL,\n  `field_area_key` int NOT NULL,\n  `location_key` int NOT NULL,\n  `company_key` int NOT NULL,\n  `arena_bombeada_nacional_tn` decimal(15,2) DEFAULT NULL,\n  `arena_bombeada_importada_tn` decimal(15,2) DEFAULT NULL,\n  `agua_inyectada_m3` decimal(15,2) DEFAULT NULL,\n  `co2_inyectado_m3` decimal(15,2) DEFAULT NULL,\n  `presion_maxima_psi` decimal(10,2) DEFAULT NULL,\n  `potencia_equipos_fractura_hp` decimal(10,2) DEFAULT NULL,\n  `cantidad_fracturas` smallint DEFAULT NULL,\n  PRIMARY KEY (`fracturing_key`),\n  UNIQUE KEY `id_base_fractura_adjiv` (`id_base_fractura_adjiv`),\n  KEY `fk_fact_fracturing_start_time` (`start_time_key`),\n  KEY `fk_fact_fracturing_end_time` (`end_time_key`),\n  KEY `fk_fact_fracturing_well` (`well_key`),\n  KEY 

# Conclusion

This notebook implemented the complete physical schema of the Oil & Gas Data Warehouse in MySQL.

The database includes:

- Six dimension tables.
- Two fact tables.
- Surrogate keys.
- Business keys.
- Foreign key relationships.
- Referential integrity constraints.

The schema is now ready for the ETL process, where data from the source datasets will be transformed and loaded into the dimensional model.